# Cybersecurity RAG Pipeline
### Retrieval-Augmented Generation over multiple security standards documents

A local RAG pipeline that answers natural-language cybersecurity questions by retrieving semantically relevant passages from a **corpus of security documents** and generating grounded answers via the Gemini API.

**Stack:** PyMuPDF · sentence-transformers (`all-MiniLM-L6-v2`) · **FAISS** · NumPy · Google Gemini 2.5
**Corpus:** OWASP Web Security Testing Guide + NIST incident-response & digital-identity guides

---

## How it works
1. Multiple PDFs are downloaded and parsed page-by-page (each chunk remembers *which document* and *which page* it came from)
2. Text is cleaned, then split into overlapping, character-budgeted chunks
3. Each chunk is embedded into a 384-dimensional vector using `all-MiniLM-L6-v2`
4. Embeddings are **cached to disk** and indexed in a **FAISS** similarity index
5. At query time, the question is embedded and FAISS returns the most relevant chunks (with **logging** of every query)
6. Retrieved chunks are passed to Gemini, which generates a grounded, source-cited answer
7. A small **evaluation** harness measures retrieval quality (Hit@k and average similarity)


## Setup
Install libraries, configure logging, and connect to the Gemini API.


In [ ]:
!pip install PyMuPDF sentence-transformers google-generativeai numpy faiss-cpu requests --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 42.5 MB/s eta 0:00:00


In [ ]:
import fitz                                              # PyMuPDF: reads PDF files
import numpy as np                                       # fast maths on arrays
import faiss                                             # fast vector similarity search
from sentence_transformers import SentenceTransformer    # converts text → vectors
import google.generativeai as genai                      # Gemini API client
import requests                                          # downloading PDFs over HTTP
import re                                                # text cleaning (regex)
import os                                                # file paths and folders
import json                                              # saving/loading chunk metadata
import glob                                              # finding files by pattern
import time                                              # timing operations for logging
import logging                                           # structured logging

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
logger = logging.getLogger("rag")
logger.setLevel(logging.INFO)
logger.handlers.clear()          # avoid duplicate handlers if the cell is re-run
logger.propagate = False

formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

# Write logs to a file (persistent) AND to the notebook output (live)
file_handler = logging.FileHandler("rag_pipeline.log")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(formatter)
logger.addHandler(console_handler)

logger.info("Logging initialised — writing to rag_pipeline.log")

2026-07-27 16:37:07,497 | INFO | Logging initialised — writing to rag_pipeline.log


In [ ]:
from google.colab import userdata

# Pull the key from Colab Secrets — never hardcode API keys in notebooks
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)

# gemini-2.5-flash-lite is the free tier model
gemini_model = genai.GenerativeModel('gemini-2.5-flash-lite')

print("\u2713 Gemini configured")

✓ Gemini configured



## Multi-Document Ingestion
Download several security PDFs into a corpus folder, extract text from all of them, and clean it. Every page remembers which document it came from.


In [ ]:
PDF_DIR = "pdfs"
os.makedirs(PDF_DIR, exist_ok=True)

# Multiple source documents — add or swap entries freely
PDF_SOURCES = {
    "owasp_wstg_v4.2.pdf":          "https://github.com/OWASP/wstg/releases/download/v4.2/wstg-v4.2.pdf",
    "nist_incident_800-61r2.pdf":   "https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-61r2.pdf",
    "nist_digital_id_800-63-3.pdf": "https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-63-3.pdf",
}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

for filename, url in PDF_SOURCES.items():
    path = os.path.join(PDF_DIR, filename)
    if os.path.exists(path):
        logger.info(f"Already downloaded: {filename}")
        continue
    try:
        logger.info(f"Downloading {filename} ...")
        resp = requests.get(url, headers=headers, timeout=60)
        resp.raise_for_status()
        with open(path, "wb") as f:
            f.write(resp.content)
        logger.info(f"Saved {filename} ({len(resp.content)/1_000_000:.1f} MB)")
    except Exception as e:
        logger.error(f"Failed to download {filename}: {e}")

downloaded = sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")))
print(f"\n\u2713 {len(downloaded)} PDF(s) available in ./{PDF_DIR}/")
for p in downloaded:
    print("   -", os.path.basename(p))

2026-07-27 16:55:23,203 | INFO | Downloading owasp_wstg_v4.2.pdf ...
2026-07-27 16:55:23,593 | INFO | Saved owasp_wstg_v4.2.pdf (10.2 MB)
2026-07-27 16:55:23,594 | INFO | Downloading nist_incident_800-61r2.pdf ...
2026-07-27 16:55:24,846 | INFO | Saved nist_incident_800-61r2.pdf (1.7 MB)
2026-07-27 16:55:24,847 | INFO | Downloading nist_digital_id_800-63-3.pdf ...
2026-07-27 16:55:25,097 | INFO | Saved nist_digital_id_800-63-3.pdf (1.6 MB)



✓ 3 PDF(s) available in ./pdfs/
   - nist_digital_id_800-63-3.pdf
   - nist_incident_800-61r2.pdf
   - owasp_wstg_v4.2.pdf


In [ ]:
def extract_text_from_pdf(pdf_path):
    doc   = fitz.open(pdf_path)
    pages = []
    for page_num in range(len(doc)):
        text = doc[page_num].get_text()
        pages.append({
            "source_file": os.path.basename(pdf_path),   # track WHICH document
            "page_number": page_num + 1,
            "text": text
        })
    doc.close()
    return pages


raw_pages = []
for pdf_path in sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf"))):
    pages = extract_text_from_pdf(pdf_path)
    raw_pages.extend(pages)
    logger.info(f"Extracted {len(pages)} pages from {os.path.basename(pdf_path)}")

n_docs = len(set(p["source_file"] for p in raw_pages))
print(f"\u2713 Extracted {len(raw_pages)} pages total from {n_docs} document(s)")

2026-07-27 16:55:34,062 | INFO | Extracted 76 pages from nist_digital_id_800-63-3.pdf
2026-07-27 16:55:34,421 | INFO | Extracted 80 pages from nist_incident_800-61r2.pdf
2026-07-27 16:55:35,579 | INFO | Extracted 465 pages from owasp_wstg_v4.2.pdf


✓ Extracted 621 pages total from 3 document(s)


In [ ]:

def clean_text(text):
    text = re.sub(r'\n{3,}', '\n\n', text)   # 3+ blank lines → 2
    text = re.sub(r'[ \t]{2,}', ' ', text)   # multiple spaces → one
    text = re.sub(r'-\n(\w)', r'\1', text)    # rejoin hyphenated words
    return text.strip()


for page in raw_pages:
    page["text"] = clean_text(page["text"])

pages = [p for p in raw_pages if len(p["text"]) > 150]

logger.info(f"Kept {len(pages)} content pages after cleaning (from {len(raw_pages)})")
print(f"\u2713 Kept {len(pages)} pages with content")

2026-07-27 16:55:40,951 | INFO | Kept 599 content pages after cleaning (from 621)


✓ Kept 599 pages with content



## Chunking



In [ ]:
def split_into_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)
    return [s.strip() for s in sentences if len(s.strip()) > 20]

In [ ]:
def make_chunks(sentences, target_chars=800, overlap_sentences=2):
    chunks = []
    i, n = 0, len(sentences)

    while i < n:
        current, char_count, j = [], 0, i

        # grow the chunk sentence-by-sentence until we hit the character budget
        while j < n and char_count < target_chars:
            current.append(sentences[j])
            char_count += len(sentences[j]) + 1
            j += 1

        if len(current) >= 2:
            chunks.append(' '.join(current))

        if j >= n:
            break

        # step back a few sentences so consecutive chunks overlap
        i = max(i + 1, j - overlap_sentences)

    return chunks

In [ ]:
all_chunks = []

for page in pages:
    sentences = split_into_sentences(page["text"])
    for chunk_text in make_chunks(sentences, target_chars=800, overlap_sentences=2):
        all_chunks.append({
            "chunk_id":    len(all_chunks),
            "source_file": page["source_file"],
            "page_number": page["page_number"],
            "text":        chunk_text
        })

logger.info(f"Created {len(all_chunks)} chunks from {len(pages)} pages")
print(f"\u2713 Created {len(all_chunks)} chunks")
print("\n--- Sample chunk ---")
print(f"Source: {all_chunks[10]['source_file']}  |  Page: {all_chunks[10]['page_number']}")
print(all_chunks[10]['text'][:300], "...")

2026-07-27 16:55:51,194 | INFO | Created 2169 chunks from 599 pages


✓ Created 2169 chunks

--- Sample chunk ---
Source: nist_digital_id_800-63-3.pdf  |  Page: 6
NIST SP 800-63-3 
 
DIGITAL IDENTITY GUIDELINES 
 
 
 
iii 
This publication is available free of charge from: https://doi.org/10.6028/NIST.SP.800-63-3 
Abstract 
These guidelines provide technical requirements for federal agencies implementing digital 
identity services and are not intended to cons ...



## Embeddings, Storage & FAISS Index
Embed every chunk, **cache the vectors to disk** so they're not recomputed each session, then load them into a **FAISS** index for fast similarity search.


In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("\u2713 Embedding model loaded")
print(f"   Dimensions: {embedding_model.get_sentence_embedding_dimension()}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Embedding model loaded
   Dimensions: 384


/tmp/ipykernel_717/1230408170.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"   Dimensions: {embedding_model.get_sentence_embedding_dimension()}")


In [ ]:
EMB_PATH    = "embeddings.npy"
CHUNKS_PATH = "chunks.json"

def build_or_load_embeddings():
    # If a cache exists AND matches the current chunk count, reuse it
    if os.path.exists(EMB_PATH) and os.path.exists(CHUNKS_PATH):
        cached = np.load(EMB_PATH)
        with open(CHUNKS_PATH) as f:
            cached_chunks = json.load(f)
        if len(cached_chunks) == len(all_chunks):
            logger.info("Loaded cached embeddings from disk (skipped recompute)")
            print(f"\u2713 Loaded {cached.shape[0]} cached embeddings")
            return cached.astype("float32"), cached_chunks
        logger.info("Cache size mismatch — recomputing embeddings")

    texts = [c["text"] for c in all_chunks]
    logger.info(f"Encoding {len(texts)} chunks")
    emb = embedding_model.encode(
        texts, batch_size=32, show_progress_bar=True, convert_to_numpy=True
    ).astype("float32")

    np.save(EMB_PATH, emb)
    with open(CHUNKS_PATH, "w") as f:
        json.dump(all_chunks, f)
    logger.info(f"Saved embeddings → {EMB_PATH} and chunks → {CHUNKS_PATH}")
    print(f"\u2713 Encoded and cached {emb.shape[0]} embeddings")
    return emb, all_chunks


embeddings, all_chunks = build_or_load_embeddings()
print(f"   Shape: {embeddings.shape}")

2026-07-27 16:56:04,433 | INFO | Encoding 2169 chunks


Batches:   0%|          | 0/68 [00:00<?, ?it/s]

2026-07-27 17:01:19,601 | INFO | Saved embeddings → embeddings.npy and chunks → chunks.json


✓ Encoded and cached 2169 embeddings
   Shape: (2169, 384)


In [ ]:
def build_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)      # inner product on normalised vectors = cosine similarity
    normalised = embeddings.copy()
    faiss.normalize_L2(normalised)      # L2-normalise in place → makes IP == cosine
    index.add(normalised)
    return index


faiss_index = build_faiss_index(embeddings)
logger.info(f"FAISS index built with {faiss_index.ntotal} vectors")
print(f"\u2713 FAISS index ready — {faiss_index.ntotal} vectors, dim {embeddings.shape[1]}")

2026-07-27 17:01:28,555 | INFO | FAISS index built with 2169 vectors


✓ FAISS index ready — 2169 vectors, dim 384



## Retrieval (FAISS + Logging)
Embed the query, search the FAISS index, and return the top matches; logging every query, its top score, latency, and which document/pages were retrieved.


In [ ]:
def retrieve(query, top_k=5):
    t0 = time.time()

    # embed the query and normalise it the same way as the index
    q = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q)

    # FAISS returns the top_k scores and their chunk indices in one call
    scores, indices = faiss_index.search(q, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        chunk = all_chunks[idx]
        results.append({
            "score":       round(float(score), 4),
            "source_file": chunk["source_file"],
            "page_number": chunk["page_number"],
            "text":        chunk["text"]
        })

    elapsed = (time.time() - t0) * 1000
    logger.info(
        f"QUERY='{query[:60]}' | top_k={top_k} | "
        f"top_score={results[0]['score']} | {elapsed:.0f}ms | "
        f"pages={[(r['source_file'], r['page_number']) for r in results]}"
    )
    return results

In [ ]:
for r in retrieve("What is SQL injection and how does it work?", top_k=3):
    print(f"[{r['score']}] {r['source_file']} p.{r['page_number']}")
    print(f"   {r['text'][:200]}...\n")

2026-07-27 17:01:36,652 | INFO | QUERY='What is SQL injection and how does it work?' | top_k=3 | top_score=0.8025 | 40ms | pages=[('owasp_wstg_v4.2.pdf', 236), ('owasp_wstg_v4.2.pdf', 236), ('owasp_wstg_v4.2.pdf', 236)]


[0.8025] owasp_wstg_v4.2.pdf p.236
   An SQL injection attack consists of insertion or “injection” of either a partial or complete SQL query via the data input or
transmitted from the client (browser) to the web application. A successful ...

[0.7481] owasp_wstg_v4.2.pdf p.236
   SQL injection attacks are a type of injection
attack, in which SQL commands are injected into data-plane input in order to affect the execution of predeﬁned SQL
commands. In general the way web applic...

[0.7239] owasp_wstg_v4.2.pdf p.236
   Web Security Testing Guide v4.2
234
Testing for SQL Injection
ID
WSTG-INPV-05
Summary
SQL injection testing checks if it is possible to inject data into the application so that it executes a user-cont...




## Generation (Gemini + Logging)
Pass retrieved chunks to Gemini as context and generate a grounded, source-cited answer; with error handling and latency logging around the API call.


In [ ]:
def build_prompt(query, retrieved_chunks):
    context_blocks = []
    for i, chunk in enumerate(retrieved_chunks):
        block = f"[Source {i+1} — {chunk['source_file']}, page {chunk['page_number']}]\n{chunk['text']}"
        context_blocks.append(block)
    context = "\n\n".join(context_blocks)

    prompt = f"""You are a cybersecurity assistant. Answer the question using ONLY the context provided below.
If the answer is not in the context, say "I don't have enough information in the document to answer that."
Always mention which source(s) you used.

--- CONTEXT ---
{context}

--- QUESTION ---
{query}

--- ANSWER ---"""
    return prompt

In [ ]:
def ask(query, top_k=5):
    retrieved = retrieve(query, top_k=top_k)
    prompt    = build_prompt(query, retrieved)

    try:
        t0 = time.time()
        response = gemini_model.generate_content(prompt)
        elapsed  = (time.time() - t0) * 1000
        logger.info(f"LLM responded in {elapsed:.0f}ms for query='{query[:60]}'")
        answer = response.text
    except Exception as e:
        logger.error(f"LLM call failed: {e}")
        print("Error contacting Gemini:", e)
        return

    print(f"Question: {query}")
    print("=" * 60)
    print(answer)
    print("\n" + "=" * 60)
    print("Sources used:")
    for r in retrieved:
        print(f"  \u2022 {r['source_file']} p.{r['page_number']} (score {r['score']})")


## Demo Queries
Example questions spanning the corpus. Because we now index both OWASP (web testing) and NIST (incident response, identity) documents, questions can be answered from whichever document is most relevant.


In [ ]:
ask("What is SQL injection and how can it be prevented?")

2026-07-27 17:01:50,941 | INFO | QUERY='What is SQL injection and how can it be prevented?' | top_k=5 | top_score=0.7702 | 35ms | pages=[('owasp_wstg_v4.2.pdf', 236), ('owasp_wstg_v4.2.pdf', 236), ('owasp_wstg_v4.2.pdf', 236), ('owasp_wstg_v4.2.pdf', 261), ('owasp_wstg_v4.2.pdf', 236)]
2026-07-27 17:01:55,845 | INFO | LLM responded in 4904ms for query='What is SQL injection and how can it be prevented?'


Question: What is SQL injection and how can it be prevented?
SQL injection is a type of injection attack where SQL commands are inserted into data-plane input to affect the execution of predefined SQL commands. This can happen when web applications mix user-supplied data with SQL syntax written by programmers. A successful SQL injection attack can allow an unauthorized user to access or manipulate data in the database, read sensitive data, modify data, execute administration operations, recover or write files on the database system, and in some cases, issue commands to the operating system. [Source 1, Source 2, Source 3]

The provided context does not contain information on how to prevent SQL injection. Therefore, I don't have enough information in the document to answer that.

Sources used:
  • owasp_wstg_v4.2.pdf p.236 (score 0.7702)
  • owasp_wstg_v4.2.pdf p.236 (score 0.7361)
  • owasp_wstg_v4.2.pdf p.236 (score 0.7054)
  • owasp_wstg_v4.2.pdf p.261 (score 0.682)
  • owasp_wstg_v4.

In [ ]:
ask("What is cross-site scripting and what are the different types?")

2026-07-27 17:01:55,878 | INFO | QUERY='What is cross-site scripting and what are the different type' | top_k=5 | top_score=0.6735 | 24ms | pages=[('nist_digital_id_800-63-3.pdf', 57), ('owasp_wstg_v4.2.pdf', 225), ('nist_digital_id_800-63-3.pdf', 57), ('owasp_wstg_v4.2.pdf', 380), ('owasp_wstg_v4.2.pdf', 320)]
2026-07-27 17:02:01,853 | INFO | LLM responded in 5975ms for query='What is cross-site scripting and what are the different type'


Question: What is cross-site scripting and what are the different types?
Cross-site Scripting (XSS) is a vulnerability that allows attackers to inject malicious code into an otherwise benign website. These scripts acquire the permissions of scripts generated by the target website and can compromise the confidentiality and integrity of data transfers between the website and client. Websites are vulnerable if they display user-supplied data from requests or forms without sanitizing the data so that it is not executable. [Source 1, Source 3]

The document mentions the following types of Cross-Site Scripting:
*   **Stored Cross-Site Scripting (XSS):** This is described as the most dangerous type of Cross Site Scripting. It occurs when a web application gathers input from a user, potentially malicious, and stores it in a data store without correctly filtering it. This stored malicious data is then presented as part of the website and runs within the user's browser under the privileges of th

In [ ]:

ask("How should an incident response team contain a security incident?")

2026-07-27 17:02:04,028 | INFO | QUERY='How should an incident response team contain a security inci' | top_k=5 | top_score=0.8018 | 23ms | pages=[('nist_incident_800-61r2.pdf', 29), ('nist_incident_800-61r2.pdf', 12), ('nist_incident_800-61r2.pdf', 29), ('nist_incident_800-61r2.pdf', 12), ('nist_incident_800-61r2.pdf', 30)]
2026-07-27 17:02:07,578 | INFO | LLM responded in 3549ms for query='How should an incident response team contain a security inci'


Question: How should an incident response team contain a security incident?
I don't have enough information in the document to answer that.

Sources used:
  • nist_incident_800-61r2.pdf p.29 (score 0.8018)
  • nist_incident_800-61r2.pdf p.12 (score 0.7823)
  • nist_incident_800-61r2.pdf p.29 (score 0.7803)
  • nist_incident_800-61r2.pdf p.12 (score 0.7776)
  • nist_incident_800-61r2.pdf p.30 (score 0.7771)



## Evaluation



In [ ]:
eval_set = [
    {"question": "What is SQL injection?",                                  "expect_keywords": ["sql", "inject", "query"]},
    {"question": "What are the types of cross-site scripting?",             "expect_keywords": ["stored", "dom", "reflect"]},
    {"question": "How do you test for session fixation?",                   "expect_keywords": ["session", "fixation", "cookie"]},
    {"question": "What is clickjacking?",                                   "expect_keywords": ["clickjack", "frame"]},
    {"question": "How should an incident response team handle an incident?","expect_keywords": ["incident", "contain", "respons"]},
]

In [ ]:
def evaluate_retrieval(eval_set, top_k=5):
    hits, total_top1, rows = 0, 0.0, []

    for item in eval_set:
        results = retrieve(item["question"], top_k=top_k)
        joined  = " ".join(r["text"].lower() for r in results)
        matched = [kw for kw in item["expect_keywords"] if kw.lower() in joined]
        hit     = len(matched) > 0

        hits       += int(hit)
        total_top1 += results[0]["score"]
        rows.append({
            "question": item["question"],
            "hit": hit,
            "matched": matched,
            "top1_score": results[0]["score"]
        })

    return hits / len(eval_set), total_top1 / len(eval_set), rows


hit_rate, avg_top1, rows = evaluate_retrieval(eval_set, top_k=5)

print("Retrieval Evaluation")
print("=" * 60)
for r in rows:
    status = "PASS" if r["hit"] else "FAIL"
    print(f"[{status}] {r['question']}")
    print(f"        matched={r['matched']}  top1_score={r['top1_score']}")
print("=" * 60)
print(f"Hit@5 (expected keyword found in retrieved chunks): {hit_rate:.0%}")
print(f"Average top-1 similarity score:                     {avg_top1:.3f}")
logger.info(f"Evaluation complete: hit_rate={hit_rate:.2f}, avg_top1={avg_top1:.3f}")

2026-07-27 17:02:19,939 | INFO | QUERY='What is SQL injection?' | top_k=5 | top_score=0.8034 | 45ms | pages=[('owasp_wstg_v4.2.pdf', 236), ('owasp_wstg_v4.2.pdf', 236), ('owasp_wstg_v4.2.pdf', 236), ('owasp_wstg_v4.2.pdf', 261), ('owasp_wstg_v4.2.pdf', 236)]
2026-07-27 17:02:19,965 | INFO | QUERY='What are the types of cross-site scripting?' | top_k=5 | top_score=0.6207 | 24ms | pages=[('nist_digital_id_800-63-3.pdf', 57), ('owasp_wstg_v4.2.pdf', 225), ('nist_digital_id_800-63-3.pdf', 57), ('owasp_wstg_v4.2.pdf', 382), ('owasp_wstg_v4.2.pdf', 320)]
2026-07-27 17:02:19,990 | INFO | QUERY='How do you test for session fixation?' | top_k=5 | top_score=0.6114 | 23ms | pages=[('owasp_wstg_v4.2.pdf', 214), ('owasp_wstg_v4.2.pdf', 213), ('owasp_wstg_v4.2.pdf', 196), ('owasp_wstg_v4.2.pdf', 198), ('owasp_wstg_v4.2.pdf', 358)]
2026-07-27 17:02:20,014 | INFO | QUERY='What is clickjacking?' | top_k=5 | top_score=0.784 | 23ms | pages=[('owasp_wstg_v4.2.pdf', 402), ('owasp_wstg_v4.2.pdf', 402), ('ow

Retrieval Evaluation
[PASS] What is SQL injection?
        matched=['sql', 'inject', 'query']  top1_score=0.8034
[PASS] What are the types of cross-site scripting?
        matched=['stored', 'dom']  top1_score=0.6207
[PASS] How do you test for session fixation?
        matched=['session', 'fixation', 'cookie']  top1_score=0.6114
[PASS] What is clickjacking?
        matched=['clickjack', 'frame']  top1_score=0.784
[PASS] How should an incident response team handle an incident?
        matched=['incident', 'respons']  top1_score=0.7384
Hit@5 (expected keyword found in retrieved chunks): 100%
Average top-1 similarity score:                     0.712


In [ ]:
# Inspect the last part of the persistent log file
print(open("rag_pipeline.log").read()[-2500:])

:01:55,845 | INFO | LLM responded in 4904ms for query='What is SQL injection and how can it be prevented?'
2026-07-27 17:01:55,878 | INFO | QUERY='What is cross-site scripting and what are the different type' | top_k=5 | top_score=0.6735 | 24ms | pages=[('nist_digital_id_800-63-3.pdf', 57), ('owasp_wstg_v4.2.pdf', 225), ('nist_digital_id_800-63-3.pdf', 57), ('owasp_wstg_v4.2.pdf', 380), ('owasp_wstg_v4.2.pdf', 320)]
2026-07-27 17:02:01,853 | INFO | LLM responded in 5975ms for query='What is cross-site scripting and what are the different type'
2026-07-27 17:02:04,028 | INFO | QUERY='How should an incident response team contain a security inci' | top_k=5 | top_score=0.8018 | 23ms | pages=[('nist_incident_800-61r2.pdf', 29), ('nist_incident_800-61r2.pdf', 12), ('nist_incident_800-61r2.pdf', 29), ('nist_incident_800-61r2.pdf', 12), ('nist_incident_800-61r2.pdf', 30)]
2026-07-27 17:02:07,578 | INFO | LLM responded in 3549ms for query='How should an incident response team contain a security


## Interactive Query
Ask anything across the loaded corpus. Retrieval spans all documents, and answers cite the specific document and page.

**Try:**
- *"How does a path traversal attack work?"* (OWASP)
- *"What are the phases of the incident response lifecycle?"* (NIST 800-61)
- *"What are the requirements for authentication assurance levels?"* (NIST 800-63)


In [ ]:
print("Ask anything about the loaded security documents.")
print("Type 'quit' to stop.\n")

while True:
    user_query = input("Your question: ").strip()
    if user_query.lower() in ("quit", "exit", "q"):
        print("Session ended.")
        break
    if not user_query:
        continue
    print()
    ask(user_query)
    print()

Ask anything about the loaded security documents.
Type 'quit' to stop.

